# Investigating TMDb Movies: Revenue and Movie Characteristics

## Introduction
This report explores the supplied TMDb movie dataset: 10,866 records and 21 columns,
covering releases from 1960 to 2015. It contains movie identifiers, descriptive
metadata, audience measures, runtime, and financial information. The analysis is
descriptive: it investigates associations in this dataset, not causes or a model
for predicting future box-office performance.

**Source:** Udacity's `Database_TMDb_movie_data/tmdb-movies.csv`, described in the
project as cleaned TMDb data from Kaggle. The adjusted financial columns are in
2010 US dollars. These are reported revenues, not profits; marketing and
distribution costs are not available.

### Questions
**How does inflation-adjusted revenue relate to a movie's budget, popularity,
average viewer rating, and runtime?**

The outcome is `revenue_adj`. The four explanatory variables are `budget_adj`,
`popularity`, `vote_average`, and `runtime`. We first examine their individual
distributions, then compare each variable with revenue. Budget quartiles and
sensitivity analyses provide additional views of the same question.

### Dataset dictionary
| Column | Meaning |
|---|---|
| id | TMDb movie identifier |
| imdb_id | IMDb identifier |
| popularity | TMDb popularity measure; its collection timing is not supplied |
| budget | Reported budget in nominal US dollars |
| revenue | Reported revenue in nominal US dollars |
| original_title | Original movie title |
| cast | Cast names, separated by pipes |
| homepage | Movie website URL |
| director | Director names |
| tagline | Promotional tagline |
| keywords | Descriptive keywords, separated by pipes |
| overview | Movie synopsis |
| runtime | Duration in minutes |
| genres | Genre labels, separated by pipes |
| production_companies | Production company names, separated by pipes |
| release_date | Supplied release-date string |
| vote_count | Number of viewer votes |
| vote_average | Average viewer rating |
| release_year | Release year |
| budget_adj | Budget adjusted to 2010 US dollars |
| revenue_adj | Revenue adjusted to 2010 US dollars |

This is a single table, so no join is needed. Pipe-separated metadata is not used
in this investigation and is left unchanged. The supplied numeric release year
is used for coverage checks; release-date strings need not be parsed here.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

def label_plot(ax, title, xlabel, ylabel):
    """Apply consistent titles, units, and readable grid lines to a chart."""
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.2)
    return ax

## Data Wrangling
### General properties
The input file remains unchanged. The following inspection checks dimensions,
column types, missing entries, duplicate records and identifiers, numeric ranges,
and financial/runtime zeros before deciding how to clean the data.

To reproduce this report, place the supplied CSV in `Database_TMDb_movie_data`
beside the notebook, or place `tmdb-movies.csv` beside it. Run all cells in order.
Only pandas, NumPy, Matplotlib, and the normal Jupyter display tools are required.

In [ ]:
data_path = Path('Database_TMDb_movie_data/tmdb-movies.csv')
if not data_path.exists():
    data_path = Path('tmdb-movies.csv')
raw_movies = pd.read_csv(data_path)
print('Rows and columns:', raw_movies.shape)
print('Release years:', raw_movies['release_year'].min(),
      'to', raw_movies['release_year'].max())
display(raw_movies.head())
raw_movies.info()

In [ ]:
display(raw_movies.isna().sum().rename('missing_entries').to_frame())
print('Exact duplicate rows:', raw_movies.duplicated().sum())
print('Repeated movie IDs:', raw_movies['id'].duplicated().sum())
numeric_columns = ['budget_adj', 'revenue_adj', 'popularity',
                   'vote_average', 'runtime', 'vote_count']
display(raw_movies[numeric_columns].describe().T)
display(raw_movies[['budget_adj', 'revenue_adj', 'runtime']]
        .eq(0).sum().rename('zero_entries').to_frame())

### Cleaning decisions
The file has one exact duplicate, also responsible for its repeated movie ID;
keep one copy so that movie is not counted twice. Budget and revenue contain
5,696 and 6,016 zero entries respectively, and runtime contains 31 zeros.
These cannot safely be interpreted as known zero cost, confirmed zero earnings,
or a valid movie duration. Nonpositive values in these three analysis fields are
therefore replaced with missing values, without changing the original CSV.

Other missing entries are in metadata outside the research question. Removing
every row missing a homepage or tagline would discard useful financial records,
so those fields are retained without imputation. Financial values are not filled
with means: doing so would manufacture budget–revenue relationships.

Use complete records for the outcome and all four explanatory variables so the
main comparisons use the same movies. This selection restricts the scope of the
results and is assessed explicitly below. Positive but unusually small financial
values are retained initially, then checked in a sensitivity analysis.

In [ ]:
movies = raw_movies.drop_duplicates().copy()
nonpositive_columns = ['budget_adj', 'revenue_adj', 'runtime']
movies[nonpositive_columns] = movies[nonpositive_columns].mask(
    movies[nonpositive_columns].le(0))
analysis_columns = ['revenue_adj', 'budget_adj', 'popularity',
                    'vote_average', 'runtime']
analysis = movies.dropna(subset=analysis_columns).copy()
# NumPy checks that values used in the analysis are finite.
finite_rows = np.isfinite(analysis[analysis_columns].to_numpy()).all(axis=1)
analysis = analysis.loc[finite_rows].copy()
print('Deduplicated rows:', len(movies))
print('Usable analysis rows:', len(analysis))
print('Excluded from main analysis:', len(movies) - len(analysis))
print(f'Retained proportion: {len(analysis) / len(movies):.2%}')
print('Remaining missing analysis values:',
      analysis[analysis_columns].isna().sum().sum())
print('Remaining duplicate records:', analysis.duplicated().sum())
display(analysis[analysis_columns].describe().T)

The cleaned table contains 10,865 unique records. The main analysis uses 3,854
movies (35.47%); 7,011 lack at least one required usable value. All retained
analysis values are finite and nonmissing. Reported adjusted budgets range from
about $0.97 to $425 million, while revenues range from about $2.37 to $2.83 billion.
The very small positive financial values are a data-quality concern, not silently
deleted observations.

To see whether the selected movies resemble the full table, compare medians of
characteristics available beyond the financial subset.

In [ ]:
coverage_columns = ['popularity', 'vote_average', 'runtime', 'release_year']
coverage = pd.DataFrame({
    'all_unique_movies_median': movies[coverage_columns].median(),
    'analysis_movies_median': analysis[coverage_columns].median()
})
display(coverage)
print('Analysis release years:', analysis['release_year'].min(),
      'to', analysis['release_year'].max())

The subset has higher median popularity (0.80 versus 0.38), a slightly higher
rating (6.2 versus 6.0), longer runtime (106 versus 99 minutes), and an earlier
median release year (2004 versus 2006). It is therefore not interchangeable with
the full dataset. All subsequent findings describe this financial subset.

## Exploratory Data Analysis
### Individual distributions
Inspect the outcome and each explanatory variable before plotting relationships.
Financial data and popularity are skewed, so base-10 logarithms make their
histograms readable. Labels identify transformed values; rating and runtime stay
in their original units. No points are removed merely to improve the plots.

In [ ]:
# Log transformations are defined only for positive values.
analysis['log_budget'] = np.log10(analysis['budget_adj'])
analysis['log_revenue'] = np.log10(analysis['revenue_adj'])
analysis['log_popularity'] = np.log10(analysis['popularity'])
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
hist_specs = [
    ('log_revenue', 'Adjusted revenue distribution', 'log10 revenue (2010 USD)'),
    ('log_budget', 'Adjusted budget distribution', 'log10 budget (2010 USD)'),
    ('log_popularity', 'Popularity distribution', 'log10 TMDb popularity'),
    ('vote_average', 'Viewer rating distribution', 'Average viewer rating'),
    ('runtime', 'Runtime distribution', 'Runtime (minutes)')
]
for ax, (column, title, xlabel) in zip(axes.flat, hist_specs):
    analysis[column].plot.hist(bins=30, ax=ax, color='#3578a8', edgecolor='white')
    label_plot(ax, title, xlabel, 'Number of movies')
axes.flat[-1].set_visible(False)
fig.tight_layout()
plt.show()

Financial values span several orders of magnitude; a raw-scale histogram would
compress most observations into its first bins. Ratings concentrate around the
middle of their observed range, and runtime has a long right tail. These shapes
motivate rank correlations and log-scaled financial comparisons rather than
assuming symmetric distributions.

### Budget and revenue
Compare inflation-adjusted budget with inflation-adjusted revenue using a
scatterplot on logarithmic axes. Compute both Spearman rank correlation and
Pearson correlation on the log-transformed financial values. These describe
association; neither estimates a causal effect.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
analysis.plot.scatter(x='budget_adj', y='revenue_adj', ax=ax,
                      alpha=0.25, s=12, color='#3578a8')
ax.set_xscale('log')
ax.set_yscale('log')
label_plot(ax, 'Movie budget and revenue: financial subset (n=3,854)',
           'Adjusted budget (2010 USD, logarithmic scale)',
           'Adjusted revenue (2010 USD, logarithmic scale)')
fig.tight_layout()
plt.show()
budget_rank_corr = analysis[['budget_adj', 'revenue_adj']].corr(
    method='spearman').iloc[0, 1]
log_money_corr = np.corrcoef(analysis['log_budget'], analysis['log_revenue'])[0, 1]
print(f'Budget/revenue Spearman correlation: {budget_rank_corr:.3f}')
print(f'Log-budget/log-revenue Pearson correlation: {log_money_corr:.3f}')

Higher-budget movies tend to have higher revenues in this subset, with substantial
scatter. Spearman correlation is 0.656; Pearson correlation of log budget and
log revenue is 0.652. A large budget does not guarantee high revenue, and the
relationship does not show that increasing a particular movie's budget would
increase its revenue.

Budget quartiles provide another view: examine the distribution of revenue within
each budget group, together with counts, medians, and means. Quartiles are defined
from the analysis subset, not from an external financial threshold.

In [ ]:
quartile_labels = ['Q1 (lowest)', 'Q2', 'Q3', 'Q4 (highest)']
analysis['budget_quartile'] = pd.qcut(analysis['budget_adj'], q=4,
                                     labels=quartile_labels)
budget_groups = analysis.groupby('budget_quartile', observed=True)['revenue_adj']
quartile_summary = budget_groups.agg(['count', 'median', 'mean'])
display(quartile_summary.rename(columns={
    'median': 'median_revenue_2010_USD', 'mean': 'mean_revenue_2010_USD'}))
fig, ax = plt.subplots(figsize=(8, 5))
analysis.boxplot(column='log_revenue', by='budget_quartile', ax=ax, grid=False)
fig.suptitle('')
label_plot(ax, 'Revenue distributions across budget quartiles',
           'Adjusted budget quartile', 'log10 adjusted revenue (2010 USD)')
fig.tight_layout()
plt.show()

Median adjusted revenue rises across the four budget quartiles: approximately
$10.82m, $41.79m, $80.28m, and $217.51m. Group sizes are 966, 961, 964, and 963.
Means exceed medians in every group, consistent with high-revenue movies pulling
the averages upward. The box plots show overlap and outliers, so quartile
membership is not a reliable guarantee of an individual movie's earnings.

### Popularity, rating, and runtime in relation to revenue
Use three separate scatterplots so each association is readable. Revenue is on a
logarithmic scale throughout; popularity is also log-scaled because of its skew.
The accompanying rank correlations put all four relationships on a common scale.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
relationship_specs = [
    ('popularity', 'Popularity and revenue', 'TMDb popularity (logarithmic scale)'),
    ('vote_average', 'Rating and revenue', 'Average viewer rating'),
    ('runtime', 'Runtime and revenue', 'Runtime (minutes)')
]
for ax, (column, title, xlabel) in zip(axes, relationship_specs):
    analysis.plot.scatter(x=column, y='revenue_adj', ax=ax,
                          alpha=0.25, s=10, color='#3578a8')
    ax.set_yscale('log')
    if column == 'popularity':
        ax.set_xscale('log')
    label_plot(ax, title, xlabel, 'Adjusted revenue (2010 USD, log scale)')
fig.tight_layout()
plt.show()
features = ['budget_adj', 'popularity', 'vote_average', 'runtime']
rank_correlations = analysis[features + ['revenue_adj']].corr(method='spearman')
display(rank_correlations.loc[features, ['revenue_adj']].rename(
    columns={'revenue_adj': 'Spearman_correlation_with_revenue'}))

Popularity has a positive rank correlation of 0.608 with revenue. Rating and
runtime have weaker positive correlations, 0.240 and 0.284 respectively, and their
scatterplots show substantial variation. Budget and popularity are more strongly
associated with revenue than rating or runtime in these separate comparisons.
This is not a multivariable model: overlapping relationships and confounding are
not controlled. Popularity and votes may themselves reflect a movie's success;
the data do not establish their measurement timing.

### Sensitivity checks
Check whether the signs depend on a few blockbusters, low-vote ratings, or tiny
reported financial values. Compare the original subset with: (1) movies with at
least 100 votes, (2) movies below or at both financial 99th percentiles, and
(3) movies with budget and revenue of at least $1,000 in adjusted dollars.
The thresholds are descriptive checks, not claims that excluded records are
incorrect. All main findings still use the original subset.

In [ ]:
high_vote_movies = analysis[analysis['vote_count'] >= 100]
trimmed_movies = analysis[
    (analysis['budget_adj'] <= analysis['budget_adj'].quantile(0.99)) &
    (analysis['revenue_adj'] <= analysis['revenue_adj'].quantile(0.99))]
financial_floor_movies = analysis[
    (analysis['budget_adj'] >= 1000) & (analysis['revenue_adj'] >= 1000)]
samples = {'Main subset': analysis, 'At least 100 votes': high_vote_movies,
           'Financial upper 1% excluded': trimmed_movies,
           'Both financial values at least $1,000': financial_floor_movies}
sensitivity_rows = []
for sample_name, sample in samples.items():
    correlations = sample[features + ['revenue_adj']].corr(method='spearman')
    row = correlations.loc[features, 'revenue_adj'].to_dict()
    row.update({'sample': sample_name, 'n': len(sample)})
    sensitivity_rows.append(row)
sensitivity = pd.DataFrame(sensitivity_rows).set_index('sample')
display(sensitivity[['n'] + features])

All four associations remain positive in these checks. In the 100-vote subset
(2,608 movies), budget and popularity correlations fall to 0.623 and 0.489, and
rating's correlation falls to 0.153. Removing the financial upper tail leaves
3,782 movies; correlations are 0.644 (budget), 0.592 (popularity), 0.224 (rating),
and 0.265 (runtime). The financial-floor check tests the small positive values
flagged during inspection. These checks support the direction of the descriptive
patterns, but do not correct the selection bias or establish statistical certainty.

## Conclusions
After removing one duplicate and treating unavailable financial/runtime values
as missing, the investigation compared 3,854 movies across four characteristics.
Individual histograms showed why financial and popularity scales needed log
transformations. Scatterplots, rank correlations, and budget-quartile box plots
then addressed the original revenue question from several angles.

- Adjusted budget and popularity have the strongest positive associations with
  adjusted revenue among the four characteristics examined (Spearman 0.656 and
  0.608).
- Average rating and runtime show weaker positive associations (0.240 and 0.284).
- Median adjusted revenue increases from $10.82m in the lowest budget quartile to
  $217.51m in the highest, but the distributions overlap substantially.
- The signs remain positive in the stated sensitivity checks; the strengths vary
  with the sample definition, particularly for rating and popularity.

### Limitations
Only 35.47% of deduplicated movies have usable values for this investigation.
The selected movies are more popular and longer on average than the full table,
so these findings must not be generalized to every movie. Zero financial values
may include genuine zero earnings as well as missing reporting; this file does
not distinguish them. Extremely small positive values also raise reporting-quality
questions. No external financial validation was performed.

The records stop in 2015 and TMDb audience measures have unspecified collection
dates. Popularity and ratings are not established as pre-release predictors.
Revenue is not profit, and the dataset omits marketing, distribution, and other
costs. Genres, release strategies, franchises, and changing markets may confound
the relationships. The comparisons are bivariate and descriptive: there are no
significance tests, causal estimates, or out-of-sample predictive evaluations.

### Further investigation
Obtain verified financial records, cost breakdowns, release territories, and
dated pre-release audience measures. Assess missingness by release year and
movie type, then examine genre and franchise differences and control for relevant
variables in a separately validated analysis. That would help determine how much
of these associations survives differences in reporting and release context.